<table style="width: 100%;">
<tr>
<td style="width: 50%; text-align: right; vertical-align: middle;">
<img src="https://github.com/gitpizzanow/dummy-files/blob/main/tp3nlp.jpg?raw=true" width="150">
</td>
<td style="width: 50%; text-align: left; vertical-align: middle;">

##  (NLP)   | TF-IDF + Cosine Similarity
> [SERIE](https://tp3-nlp-ing4.netlify.app/)


* *Document Frequency (DF)*
* *IDF + Smoothing*
* *TF (Term Frequency)*
* *Cosine Similarity*



</td>
</tr>
</table>

>Data: Wikipedia-like dataset

In [2]:
from sklearn.datasets import fetch_20newsgroups

docs = fetch_20newsgroups(
    subset='train',
    remove=('headers', 'footers', 'quotes')
).data[:3000]

In [3]:
type(docs)

list

In [4]:
docs[10]

'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'

![STEP 1 - Preprocessing](https://img.shields.io/badge/STEP%201%20-%20Preprocessing-blue)

In [ ]:
def preprocess_text(text):
 
    text = text.lower()
    text = ''.join(char for char in text if char.isalnum() or char.isspace())
    tokens = text.split()
    return tokens


![STEP 2 - Vocabulary](https://img.shields.io/badge/STEP%202%20-%20Vocabulary-blue)

In [6]:
def build_vocab(docs):
    vocab = set()
    for doc in docs:
        tokens = preprocess_text(doc)
        vocab.update(tokens)
    return sorted(list(vocab))

[![STEP 3 DF](https://img.shields.io/badge/STEP_3_DF-Document_Frequency-pink)](https://digitalpro.dev)

In [ ]:
def compute_df(docs, vocab):
    preprocessed_docs = [set(preprocess_text(doc)) for doc in docs]
    df = {}
    for word in vocab:
        df[word] = sum(1 for tokens in preprocessed_docs if word in tokens)
    return df

[![STEP 4 IDF](https://img.shields.io/badge/STEP_4_IDF-Inverse_Document_Frequency-pink)](https://digitalpro.dev)

In [9]:
import math
def compute_idf(df, N):
    idf = {}
    for word, freq in df.items():
        idf[word] = math.log(N / freq)
    return idf

[![STEP 5 TF Vector](https://img.shields.io/badge/STEP_5_TF-Term_Frequency_Vector-pink)](https://digitalpro.dev)

In [ ]:
from collections import Counter
def compute_tf(doc, vocab):
    tokens = preprocess_text(doc)
    total = len(tokens)
    counts = Counter(tokens)
    tf = {}
    for word in vocab:
        tf[word] = counts[word] / total if total > 0 else 0
    return tf

[![STEP 6 TF-IDF Matrix](https://img.shields.io/badge/STEP_6_TFIDF-Build_TF_IDF_Matrix-pink)](https://digitalpro.dev)

In [ ]:
import numpy as np
from collections import Counter
def build_tfidf(docs):
    tokenized_docs = [preprocess_text(doc) for doc in docs]
    vocab = sorted({word for tokens in tokenized_docs for word in tokens})
    N = len(docs)
    df = {word: sum(1 for tokens in tokenized_docs if word in set(tokens)) for word in vocab}
    idf = compute_idf(df, N)
    tfidf_matrix = []
    for tokens in tokenized_docs:
        counts = Counter(tokens)
        total = len(tokens)
        vec = [(counts[word] / total if total > 0 else 0) * idf[word] for word in vocab]
        tfidf_matrix.append(vec)
    return np.array(tfidf_matrix), vocab

[![STEP 7 Cosine Similarity](https://img.shields.io/badge/STEP_7-Cosine_Similarity-pink)](https://digitalpro.dev)

In [12]:
def cosine(a, b):
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0
    return dot / (norm_a * norm_b)

[![STEP 8 Search Engine](https://img.shields.io/badge/STEP_8-Search_Engine_REAL_VERSION-orange)](https://digitalpro.dev)

In [ ]:
import numpy as np
from collections import Counter
def search(query, docs, top_k=5):
    tfidf_matrix, vocab = build_tfidf(docs)

    q_words = preprocess_text(query)
    q_counts = Counter(q_words)
    q_vec = np.array([q_counts[w] / len(q_words) if len(q_words) > 0 else 0 for w in vocab])

    scores = []

    for i, doc_vec in enumerate(tfidf_matrix):
        score = cosine(q_vec, doc_vec)
        scores.append((score, i))

    scores.sort(reverse=True)

    return scores[:top_k]

> TEST

In [ ]:
query = "machine learning neural network"
results = search(query, docs)

for score, idx in results:
    print(score)
    print(docs[idx][:200])
    print("-" * 50)

KeyboardInterrupt: 